# Bayesian spatio-temporal Poisson model

Death counts by township x age group x year, with ICAR spatial effects (intercept and slope),
random-walk age and time effects and their interactions. Sampled with nutpie (JAX backend),
4 chains x 1000 draws. Run once for each sex (`SEX = 1` male, `SEX = 2` female).

Working directory: `code/`

- Input: `data/restricted/DeathCount_v5.csv` (HWDC, not distributed), `data/NB_mat.csv`
- Output: `output/TraceSummary_{m,f}.csv`, `output/posterior/PPmean_{sex}.csv`, `output/posterior/PPd{sex}_{chain}.csv.gz`


## Environment

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import nutpie
import jax
print(f'Running on PyMC v{pm.__version__}')
print(jax.default_backend(), jax.devices())
az.style.use("arviz-darkgrid")

os.makedirs('../output/posterior', exist_ok = True)

## Parameter

In [ ]:
SEX = 1  # 1 = male, 2 = female
sex_eng = {1: 'male', 2: 'female'}[SEX]
sex_abb = {1: 'm', 2: 'f'}[SEX]

## Data

In [ ]:
DeathCount = pd.read_csv('../data/restricted/DeathCount_v5.csv', encoding = 'latin1')
# Only estimate TAIWAN REGION data
# That means Kingmen, Lianjiang would not be concluded
Data = DeathCount[(DeathCount['SEX'] == SEX) & (DeathCount['Urbanicity_AST2023'] != 9)]

# lookup
lookup_moi = dict(zip(Data.MOI.unique(), range(len(Data.MOI.unique()))))
lookup_moi = Data.MOI.replace(lookup_moi).values

lookup_agp = dict(zip(Data.AGP.unique(), range(len(Data.AGP.unique()))))
lookup_agp = Data.AGP.replace(lookup_agp).values

lookup_year = dict(zip(Data.YEAR.unique(), Data.YEAR.unique() - 2010))
lookup_year = Data.YEAR.replace(lookup_year).values

In [ ]:
# Township adjacency (same township order as Data.MOI.unique())
NB = pd.read_csv('../data/NB_mat.csv', encoding = 'latin1')
NB_array = np.array(NB)

## Model

In [ ]:
with pm.Model() as Model1:
    # Data
    YEAR = pm.Data('YEAR', lookup_year)
    AGP = pm.Data('AGP', lookup_agp)
    COUNT = pm.Data('COUNT', Data.COUNT)
    POP = pm.Data('POP', Data.POP)

    N_s = len(Data.MOI.unique())
    N_a = len(Data.AGP.unique())
    N_t = len(Data.YEAR.unique())
    
    # Priors
    alpha_0 = pm.Normal('alpha_0', mu = 0, sigma = 100000)
    beta_0 = pm.Normal('beta_0', mu = 0, sigma = 100000)

    sigma_alpha_s = pm.HalfNormal('sigma_alpha_s', sigma = 1)
    alpha_s = pm.ICAR("alpha_s", W=NB_array, sigma=sigma_alpha_s)
    
    sigma_beta_s = pm.HalfNormal('sigma_beta_s', sigma = 1)
    beta_s = pm.ICAR("beta_s", W=NB_array, sigma=sigma_beta_s)
    
    # Agp Terms
    sigma_alpha_a = pm.HalfNormal('sigma_alpha_a', sigma = 1)
    alpha_a_raw = pm.GaussianRandomWalk('alpha_a_raw', mu = 0, sigma = sigma_alpha_a, init_dist = pm.Normal.dist(0, 1), shape = N_a)
    alpha_a = pm.Deterministic('alpha_a', alpha_a_raw - alpha_a_raw[0])
    
    # Time Terms
    sigma_alpha_t = pm.HalfNormal('sigma_alpha_t', sigma = 1)
    alpha_t_raw = pm.GaussianRandomWalk('alpha_t_raw', mu = 0, sigma = sigma_alpha_t, init_dist = pm.Normal.dist(0, 1), shape = N_t)
    alpha_t = pm.Deterministic('alpha_t', alpha_t_raw - alpha_t_raw[0])

    # Interaction
    sigma_nu = pm.HalfNormal('sigma_nu', sigma = 1)
    nu_raw = pm.GaussianRandomWalk('nu_raw', mu = 0, sigma = sigma_nu, 
                               init_dist = pm.Normal.dist(0, 1), shape = (N_s, N_t))
    nu = pm.Deterministic('nu', nu_raw - nu_raw[:, 0][:, None])
    
    
    sigma_gamma = pm.HalfNormal('sigma_gamma', sigma = 1)
    gamma_raw = pm.GaussianRandomWalk('gamma_raw', mu = 0, sigma = sigma_gamma, 
                               init_dist = pm.Normal.dist(0, 1), shape = (N_a, N_t))
    gamma = pm.Deterministic('gamma', gamma_raw - gamma_raw[:, 0][:, None])

    sigma_xi = pm.HalfNormal('sigma_xi', sigma = 1)
    xi = pm.Normal('xi', mu = 0, sigma = sigma_xi, shape = (N_a, N_s))
    
    # Log Mean Model
    log_m = alpha_0 + beta_0 * YEAR
    log_m += alpha_s[lookup_moi] + beta_s[lookup_moi] * YEAR
    log_m += alpha_a[lookup_agp] + alpha_t[lookup_year]
    log_m += nu[lookup_moi, lookup_year] + gamma[lookup_agp, lookup_year] + xi[lookup_agp, lookup_moi]

    # Likelihood
    mu = pm.math.exp(log_m) * POP
    d = pm.Poisson('d', mu=mu, observed=COUNT)

## Sampling

In [ ]:
Compiled_model = nutpie.compile_pymc_model(Model1, backend='jax', gradient_backend='jax')
Trace_Model1 = nutpie.sample(
        Compiled_model,
        chains=4,
        draws=1000,
        tune=100000,
        target_accept=0.8,
        seed=1011
    )

## Diagnostics

In [ ]:
var_names = ['alpha_0', 'beta_0', 'alpha_s', 'beta_s', 'alpha_a', 'alpha_t', 'nu', 'gamma', 'xi']
Summary_Model1 = az.summary(Trace_Model1, var_names = var_names, round_to = 2, stat_focus = "mean")
Summary_Model1.to_csv(f'../output/TraceSummary_{sex_abb}.csv', index = True)

In [ ]:
Trace_plot = az.plot_trace(Trace_Model1, var_names = var_names)

In [ ]:
Trace_Model1.sample_stats["diverging"].sum().values

In [ ]:
with Model1:
    pm.compute_log_likelihood(Trace_Model1)
Loo_Model1 = az.loo(Trace_Model1)
WAIC_Model1 = az.waic(Trace_Model1)
print(Loo_Model1)
print(WAIC_Model1)

## Posterior predictive

In [ ]:
with Model1:
    PosteriorPredict_Model1 = pm.sample_posterior_predictive(Trace_Model1)
PP = PosteriorPredict_Model1['posterior_predictive']

In [ ]:
# Posterior mean of the death counts
PPmean = PP['d'].mean(dim=['chain', 'draw'])
PPmean_df = PPmean.to_dataframe()
PPmean_df.to_csv(f'../output/posterior/PPmean_{sex_eng}.csv', index = False)

In [ ]:
# Every draw of the death counts, one file per chain
# Rows within each draw follow the row order of Data
PPd_dataframe = PP['d'].to_dataframe().reset_index()
for chain in range(4):
    PPd_chain = PPd_dataframe[(PPd_dataframe['chain'] == chain)]
    output_filename = f'../output/posterior/PPd{sex_eng}_{chain}.csv.gz'
    PPd_chain.to_csv(output_filename, index=False, compression='gzip')
    print(f'Saved: {output_filename}')